# Haircut diagnostics

Decides whether SFTDS haircuts carry enough variation to support a fund / dealer / collateral fixed-effects decomposition, *before* any regression is run.

**Unit:** one relationship-day = `(date, fund, dealer, bond, leg)`, with a volume-weighted haircut and repo rate. Same universe as the main panels (non-cleared EUR `SPEC` repo, `S1311` collateral, DE/IT bonds, Cayman-`IF` funds, 2021-01-04 to 2025-11-01). `repo_rate` is pulled alongside so its variation can be compared directly with the haircut's.

**Caveats:** the dealer is just the non-fund counterparty, so fund-to-fund trades (a Cayman IF on both legs) are not excluded; `repo_rate` is assumed to be the column name in the source table.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
DATA = 'C:/Users/hermesf/Projects/JobMarket/Data'   # forward slashes work on Windows

In [ ]:
# Bond universe: DE/IT, still alive, priced (same filter as the build notebooks)
bonds = pd.read_csv(f'{DATA}/bond_timeseries_v2.csv',
                    usecols=['Dates', 'ISIN', 'PX_ASK', 'bond_maturity'])
bonds['Dates'] = pd.to_datetime(bonds['Dates'])
bonds['bond_maturity'] = pd.to_datetime(bonds['bond_maturity'])
bonds = bonds[((bonds['bond_maturity'] - bonds['Dates']).dt.days / 365) > 0]
bonds = bonds[bonds['ISIN'].str[:2].isin(['DE', 'IT']) & bonds['PX_ASK'].notna()]
securities = tuple(bonds['ISIN'].unique())
print(len(securities), 'bonds')

In [ ]:
# Pull both legs at the relationship-day grain with volume-weighted haircut & repo rate
def leg_query(fund_side):
    if fund_side == 'borrower':   # HF posts bond to finance a long
        fund, dealer, ctry, enr, leg = 'borrower_id', 'lender_id', 'borrower_country_residence', 'borrower', 'long'
    else:                         # HF lends cash to source a bond -> short
        fund, dealer, ctry, enr, leg = 'lender_id', 'borrower_id', 'lender_country_residence', 'lender', 'short'
    return f"""
SELECT s.business_date,
       s.security_isin AS isin,
       s.{fund}   AS fund_id,
       s.{dealer} AS dealer_id,
       '{leg}' AS leg,
       SUM(s.nominal_euro) AS nominal_euro,
       SUM(s.haircut   * s.nominal_euro) / NULLIF(SUM(CASE WHEN s.haircut   IS NOT NULL THEN s.nominal_euro ELSE 0 END), 0) AS haircut,
       SUM(s.repo_rate * s.nominal_euro) / NULLIF(SUM(CASE WHEN s.repo_rate IS NOT NULL THEN s.nominal_euro ELSE 0 END), 0) AS repo_rate
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_{enr}
       ON s.{fund} = s_{enr}.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'non-cleared'
  AND s.{ctry} = 'KY'
  AND s_{enr}.sector = 'IF'
  AND s.gnlcoll = 'SPEC'
  AND s.assttp_scty_issr_sector_riad = 'S1311'
  AND s.nominal_euro > 0
  AND s.security_isin IN {securities}
GROUP BY s.business_date, s.security_isin, s.{fund}, s.{dealer}
"""

df = pd.concat([pd.read_sql_query(leg_query('borrower'), cnxn),
                pd.read_sql_query(leg_query('lender'), cnxn)], ignore_index=True)
df['date'] = pd.to_datetime(df['business_date'])
df['collateral_country'] = df['isin'].str[:2]
df.to_csv(f'{DATA}/haircut_cells.csv', index=False)   # reused by the regression step
print(df.shape)

## A. Distribution — compression and the mass at zero

Share of relationship-days with the haircut (and repo rate) exactly zero / negative / positive, plus quantiles, split by leg and country. This is where the spike at zero and the long-vs-short sign difference show up.

In [ ]:
def dist(var):
    d = df.dropna(subset=[var])
    stats = (d.groupby(['leg', 'collateral_country'])[var]
               .agg(n='size', mean='mean', sd='std',
                    p25=lambda x: x.quantile(.25), p50='median',
                    p75=lambda x: x.quantile(.75)))
    flags = (d.assign(zero=d[var].abs() < 1e-6, neg=d[var] < -1e-6, pos=d[var] > 1e-6)
               .groupby(['leg', 'collateral_country'])[['zero', 'neg', 'pos']].mean())
    return stats.join(flags).round(4)

print('HAIRCUT');   display(dist('haircut'))
print('REPO_RATE'); display(dist('repo_rate'))

## B. Within-cell variation — is each effect identified?

Hold two dimensions (plus `leg`) fixed; the residual haircut variation identifies the fixed effect of the dimension that **varies**:

- vary funds within dealer x collateral -> **fund effect** (fund bargaining power / risk)
- vary dealers within fund x collateral -> **dealer effect** (dealer market power)
- vary collateral within fund x dealer -> **collateral effect** (collateral risk)

The effect belongs to the dimension that varies, not the one held fixed (the reverse of the original sketch). `incl_date=True` adds the day to the cell — the within-day case the FE model relies on; `False` pools over time and so also absorbs cross-time variation, making it an upper bound.

`within_var_share` = within-cell share of haircut variance among cells with >=2 counterparties (what is left for that effect after the conditioning). `share_obs_varying` = share of all relationship-days sitting in a cell with >=2 counterparties *and* a non-degenerate haircut. If both are near zero, that effect is not estimable.

In [ ]:
def within_diag(var, keys, label, incl_date, tol=1e-6):
    k = keys + (['date'] if incl_date else [])
    d = df.dropna(subset=[var])
    g = d.groupby(k)[var]
    size, cmean, csd = g.transform('size'), g.transform('mean'), g.transform('std')
    multi = size >= 2
    varying = multi & (csd.fillna(0) > tol)
    wss = ((d[var][multi] - cmean[multi]) ** 2).sum()
    tss = ((d[var][multi] - d[var][multi].mean()) ** 2).sum()
    return {'var': var, 'effect': label, 'incl_date': incl_date,
            'n_obs': len(d), 'n_cells': g.ngroups, 'obs_in_multi': int(multi.sum()),
            'share_obs_varying': round(float(varying.mean()), 4),
            'within_var_share': round(wss / tss, 4) if tss > 0 else np.nan,
            'cell_sd_p50': round(float(csd[multi].median()), 5),
            'cell_sd_p90': round(float(csd[multi].quantile(.9)), 5)}

effects = [('fund (bargaining/risk)', ['dealer_id', 'isin', 'leg']),
           ('dealer (market power)',  ['fund_id', 'isin', 'leg']),
           ('collateral (risk)',      ['fund_id', 'dealer_id', 'leg'])]
rows = [within_diag(var, keys, label, incl)
        for var in ['haircut', 'repo_rate']
        for label, keys in effects
        for incl in (True, False)]
display(pd.DataFrame(rows))

## C. Network connectivity & mobility

The fund and dealer effects are only **jointly** identified within a connected component of the fund-dealer trading graph, and only if funds trade with several dealers and dealers with several funds. This reports mobility and the size of the largest connected component (dependency-free union-find).

In [ ]:
print('funds  :', df['fund_id'].nunique())
print('dealers:', df['dealer_id'].nunique())
print('bonds  :', df['isin'].nunique())
print('dates  :', df['date'].nunique())
print('cells  :', len(df))

dpf = df.groupby('fund_id')['dealer_id'].nunique()
fpd = df.groupby('dealer_id')['fund_id'].nunique()
print('\nfunds with >=2 dealers :', int((dpf >= 2).sum()), '/', dpf.size, '| median dealers/fund:', int(dpf.median()))
print('dealers with >=2 funds :', int((fpd >= 2).sum()), '/', fpd.size, '| median funds/dealer:', int(fpd.median()))

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

for f, d in df[['fund_id', 'dealer_id']].drop_duplicates().itertuples(index=False):
    union(('F', f), ('D', d))
root = df['fund_id'].map(lambda f: find(('F', f)))
sizes = df.groupby(root).size().sort_values(ascending=False)
top = sizes.index[0]
print('\nconnected components:', sizes.size)
print('largest covers {:.1%} of cells | {} funds | {} dealers'.format(
    sizes.iloc[0] / len(df),
    df.loc[root == top, 'fund_id'].nunique(),
    df.loc[root == top, 'dealer_id'].nunique()))